In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
np.random.seed(42)

# Parameters of the grid
shape        = (2000, 2000)
xrange       = (-4, 4)
yrange       = (-3, 5)

# Parameters of the airfoil
mu    = -0.15 + 0.15j
cut_h = 0.07
V_inf = 3
alpha = 30 * (np.pi / 180)
R     = np.sqrt((1 - mu.real)**2 + mu.imag**2)
gamma = 4 * np.pi * V_inf * R * np.sin( alpha + np.arcsin(mu.imag / R) )
rot   = np.exp(1j * alpha)

# Inverse Joukowsky transform
def z2zeta(z):
    discriminant = np.sqrt(z**2 - 4)
    zeta = (z + discriminant) / 2
    in_unit_circle = (np.abs(zeta) <= 1)
    below_wing = (z.real**2 < 4) & (z.imag > 0) & \
                 (z.imag < cut_h * (4 - z.real**2))
    bad_branch = below_wing ^ in_unit_circle
    zeta[bad_branch] -= discriminant[bad_branch]
    return zeta

# W = vx - i vy
def complex_velocity(z):
    zeta     = z2zeta(z)
    W_tilde  = V_inf * np.exp(-1j * alpha) 
    W_tilde += 1j * gamma / (2 * np.pi * (zeta - mu))
    W_tilde -= V_inf * R**2 * np.exp(1j * alpha) / (zeta - mu)**2
    W = W_tilde / (1 - 1/zeta**2)
    return W.conj()

# Bernoullis's principle
def pressure(z):
    st = complex_velocity(z)
    P = 0.5 * (V_inf**2 - np.abs(st)**2)
    return P

# Define the complex plane grid
rez  = np.linspace(*xrange, shape[0])[:, None]
imz  = np.linspace(*yrange, shape[1])[None, :]
z    = (rez + 1j * imz)

# Joukowsky airfoil profile
theta     = np.linspace(0, 2*np.pi, 100)
zeta_circ = mu + R * np.exp(1j * theta)
z_airfoil = (zeta_circ + 1/zeta_circ) * rot.conj()

##### Do the plotting #####

fig, ax = plt.subplots(figsize = (6,6))
ax.axis('off')

ax.imshow(pressure(z * rot).T, origin = 'lower', cmap = 'jet', extent = xrange + yrange, vmin = -10, vmax = 7)

W = complex_velocity(z * rot) / rot
ax.streamplot(z.real.T, z.imag.T, W.real.T, W.imag.T, color = 'k', linewidth = 0.7)
ax.fill(z_airfoil.real, z_airfoil.imag, color = 'white', zorder=2)
ax.set_xlim(*xrange)
ax.set_ylim(*yrange)

'''
ax.arrow(-0.5, +0.5, 0, 3, color = 'k', width = .07, zorder=3)
ax.arrow(-0.5, +0.5, 2, 0, color = 'k', width = .07, zorder=3)
ax.text(-0.2, 3.5, r'$F_s$', fontdict = {'size': 25, 'color': 'k'})
ax.text(1.5, 0.8, r'$F_{arr}$', fontdict = {'size': 25, 'color': 'k'})
'''

plt.savefig('AngleAtac30.pdf', dpi = 200, bbox_inches='tight', pad_inches = 0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
np.random.seed(42)

# Parameters of the grid
shape        = (2000, 2000)
xrange       = (-3, 3)
yrange       = (-2, 4)

# Parameters of the airfoil
mu    = 0.0
V_inf = 3
alpha = 30 * (np.pi / 180)
R     = 1.5 * np.sqrt((1 - mu.real)**2 + mu.imag**2)
gamma = 4 * np.pi * V_inf * R * np.sin( alpha + np.arcsin(mu.imag / R) )
rot   = np.exp(1j * alpha)

# W = vx - i vy
def complex_velocity(z):
    zeta     = z
    W_tilde  = V_inf * np.exp(-1j * alpha) 
    W_tilde += 1j * gamma / (2 * np.pi * (zeta - mu))
    W_tilde -= V_inf * R**2 * np.exp(1j * alpha) / (zeta - mu)**2
    W = W_tilde
    return W.conj()

# Bernoullis's principle
def pressure(z):
    st = complex_velocity(z)
    P = 0.5 * (V_inf**2 - np.abs(st)**2)
    return P

# Define the complex plane grid
rez  = np.linspace(*xrange, shape[0])[:, None]
imz  = np.linspace(*yrange, shape[1])[None, :]
z    = (rez + 1j * imz)

# Joukowsky airfoil profile
theta     = np.linspace(0, 2*np.pi, 100)
zeta_circ = mu + R * np.exp(1j * theta)
z_airfoil = zeta_circ * rot.conj()

##### Do the plotting #####

fig, ax = plt.subplots(figsize = (6,6))
ax.axis('off')

ax.imshow(pressure(z * rot).T, origin = 'lower', cmap = 'jet', extent = xrange + yrange, vmin = -10, vmax = 7, alpha = 0.5)

W = - complex_velocity(z * rot) / rot
ax.streamplot(z.real.T, z.imag.T, W.real.T, W.imag.T, color = 'k', linewidth = 1)
ax.fill(z_airfoil.real, z_airfoil.imag, color = 'gray', zorder=2)
ax.set_xlim(*xrange)
ax.set_ylim(*yrange)

'''
ax.arrow(0, 0, 3, 0, color = 'k', width = .07, zorder=3)
ax.text(2.5, 0.6, r'$F_{Stokes}$', fontdict = {'size': 25, 'color': 'k'})
'''

ax.arrow(0, 0, 0, 2.5, color = 'k', width = .07, zorder=3)
ax.add_patch(patches.FancyArrowPatch((0.8, -0.8), (0.8, 0.8), connectionstyle = "arc3,rad=0.35", color ='darkblue',
                                     arrowstyle = "Simple, tail_width=5, head_width=15, head_length=10", zorder=3))
ax.add_patch(patches.FancyArrowPatch((-0.8, 0.8), (-0.8, -0.8), connectionstyle = "arc3,rad=0.35", color ='darkblue',
                                     arrowstyle = "Simple, tail_width=5, head_width=15, head_length=10", zorder=3))
ax.text(0.5, 3.0, r'$F_{Magnus}$', fontdict = {'size': 25, 'color': 'k'})


plt.savefig('EfecteMagnus.pdf', dpi = 200, bbox_inches='tight', pad_inches = 0)

In [ ]:
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.animation as animation

shape = (400, 100)
iters = 4000
rho0  = 100
tau   = 0.55 # 0.75
frame_every = 8
np.random.seed(42)

x     = np.arange(shape[0])[:, None]
y     = np.arange(shape[1])[None, :]
curl  = np.zeros(shape + (iters//frame_every,)) 

D2Q9  = np.array([[di, dj, 4 / (9 * 4**(di**2 + dj**2))] for di in (-1,0,1) for dj in (-1,0,1)])
stone = (x - 50)**2 + (y - 50)**2 < 10**2

# Initial conditions
F = np.ones(shape + (9,)) + 0.01 * np.random.randn(*shape, 9)
F[:,:,7] += 2 * (1 + 0.2 * np.cos(2 * np.pi * x / shape[0] * 4))
F *= rho0 / np.sum(F, 2)[..., None]

print('Evolving fluid...')
for it in tqdm(range(iters)):
    
    # Advection
    for k, (di, dj, w) in enumerate(D2Q9):
        F[:,:,k] = np.roll(F[:,:,k], int(di), axis = 0)
        F[:,:,k] = np.roll(F[:,:,k], int(dj), axis = 1)
    
    # Save stone
    stoneF = F[stone,:]
    
    # Hydro variables
    rho = np.sum(F, 2)
    ux  = np.sum(F * D2Q9[:, 0], 2) / rho
    uy  = np.sum(F * D2Q9[:, 1], 2) / rho
    
    # Collisions
    Feq = rho[..., None] * D2Q9[:, 2] * (1 + 3 * (D2Q9[:, 0] * ux[..., None] + D2Q9[:, 1] * uy[..., None])    +  \
                                         (9/2) * (D2Q9[:, 0] * ux[..., None] + D2Q9[:, 1] * uy[..., None])**2 -  \
                                         (3/2) * (ux[..., None]**2 + uy[..., None]**2))
    F += - (1 / tau) * (F - Feq)
    
    # Apply boundaries 
    F[stone,:] = stoneF[:, ::-1]

    #Compute vorticity
    if it % frame_every == 0:
        dvx_dy = np.roll(ux, -1, axis = 1) - np.roll(ux, 1, axis = 1)
        dvy_dx = np.roll(uy, -1, axis = 0) - np.roll(uy, 1, axis = 0)
        curl[..., it//frame_every] =  dvx_dy - dvy_dx

##### Do the plotting

fig, ax = plt.subplots(frameon = False, figsize = (16, 4))
ax.axis('off')

ish = ax.imshow(curl[..., -1].T, cmap='ocean', interpolation = 'bicubic', origin = 'lower', vmin = -.04, vmax = .04)

stoneshape = 50 + 50j + 12 * np.exp(1j * np.linspace(0, 2*np.pi, 100))
ax.streamplot(x.T, y.T, ux.T, uy.T, color = 'k', linewidth = 1.1)
ax.fill(stoneshape.real, stoneshape.imag, color = 'wheat', zorder = 2)

plt.savefig('FluxTurbulent.pdf', dpi = 200, bbox_inches='tight', pad_inches = 0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x1 = np.linspace(0, 3, 100)

fig, ax = plt.subplots(figsize = (10, 10))
plt.rc('font', size = 18)

ax.plot(x1, 1 +  x1**2, 'b', lw = 3)
ax.plot([0, 1.5], [18.3, 3.3], 'b', lw = 3)
ax.plot([1.5, 3], [3.3, 10], 'ok', ms = 10)
ax.plot([1.23, 2.23], [6, 6], 'ok', ms = 7)

ax.plot([1.23, 1.23], [6, 0.0], 'k:', lw = 2)
ax.plot([0, 1.5], [3.3, 3.3], 'k:', lw = 2)
ax.plot([1.5, 1.5], [3.3, 0.0], 'k:', lw = 2)
ax.plot([0, 2.25], [6.0, 6.0], 'k:', lw = 2)
ax.plot([2.23, 2.23], [6.0, 0.0], 'k:', lw = 2)
ax.plot([0, 3], [10.0, 10.0], 'k:', lw = 2)
ax.plot([3, 3], [10.0, 0.0], 'k:', lw = 2)

ax.text(2.75, 10.5, 'Punt crític')
ax.text(1.65, 2.8,  ' Punt\n triple')
ax.text(1.6, 6.3, "Punt normal\n d'ebullició")
ax.text(0.4, 6.3, "Punt normal\n   de fusió")

ax.text(0.6, 4.5, "GEL", color = 'blue', fontsize = 18)
ax.text(1.5, 8.0, "LÍQUID", color = 'blue', fontsize = 18)
ax.text(2.5, 2.0, "VAPOR", color = 'blue', fontsize = 18)

ax.set_xticks([1.23, 1.5, 2.23, 3])
ax.set_xticklabels([0, 0.01, 100, 373.99])
ax.set_yticks([3.3, 6, 10])
ax.set_yticklabels([0.006, 1, 217.75])

ax.set_xlabel(r'Temperatura ($^\circ C$)')
ax.set_ylabel(r'Pressió (atm)')

ax.set_xlim(0, 3.5)
ax.set_ylim(0, 13)

plt.savefig('DiagramaDeFases.pdf', dpi = 200, bbox_inches='tight', pad_inches = 0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize = (10, 5))
plt.rc('font', size = 15)

ax.plot([-1, 3, 8, 12, 20, 24], 
        [0, 1, 1, 2,  2,  3 ], 'r', lw = 3)
ax.plot([-1, 3], [1.0, 1.0], 'k:', lw = 2)
ax.plot([-1, 12], [2.0, 2.0], 'k:', lw = 2)

ax.text(0, 0.5, "Gel", rotation = 45)
ax.text(3.5, 1.1, "Gel + Líquid")
ax.text(8.5, 1.4, "Líquid", rotation = 45)
ax.text(13.5, 2.1, "Líquid + Vapor")
ax.text(20.5, 2.4, "Vapor", rotation = 45)

ax.set_xticks([])
ax.set_xticklabels([])
ax.set_yticks([1, 2])
ax.set_yticklabels([0, 100])

ax.set_xlim(-1, 25)
ax.set_ylim(0, 3)

ax.set_xlabel(r'Calor subministrada')
ax.set_ylabel(r'Temperatura ($^\circ C$)')

plt.savefig('TvsQ.pdf', dpi = 200, bbox_inches='tight', pad_inches = 0)

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

lam = np.linspace(10, 2000, 100)
cmap = mpl.colormaps['jet']

def Planck(lam, T):
    h, c, k = 6.62607015e-34, 2.99792458e8, 1.380649e-23 
    lam_m   = lam * 1e-9
    result  = (2 * h * c**2 / lam_m**5) / (np.exp(h * c / (lam_m * k * T)) - 1)
    return result / 1e13
    
fig, ax = plt.subplots(figsize = (10, 8))
plt.rc('font', size = 15)

for T in [4000, 5000, 6000]:
    pl      = Planck(lam, T)
    ax.plot(lam, pl, 'k')
    ax.text(lam[np.argmax(pl)] - 170, np.max(pl) + 0.1, f"T = {T} K")

for lam in np.linspace(400, 750, 70):
    ax.plot(2 * [lam], [0.1, 3.5], color = cmap(int( 255 * (lam - 400) / 350 )), alpha = 0.7, zorder = 1)

ax.set_xlim(0, 2000)
ax.set_ylim(0.1, 3.5)

ax.set_yticks([])

ax.set_xlabel("Longitud d'ona (nm)")
ax.set_ylabel("Intensitat")

plt.savefig('EspectreDePlanck.pdf', dpi = 200, bbox_inches='tight', pad_inches = 0)

In [ ]:
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.animation as animation

shape        = (1000, 1000)
xrange       = (-1, +1)
yrange       = (-1, +1)

x = np.linspace(*xrange, shape[0])[:, None]
y = np.linspace(*yrange, shape[1])[None, :]
canvas = np.ones(shape + (3,))

r1 = np.sqrt( (x + 0.5)**2 + y**2 )
r2 = np.sqrt( (x - 0.5)**2 + y**2 )

f  = np.exp(8j * np.pi * r1) + np.exp(8j * np.pi * r2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (10, 5))
plt.rc('font', size = 15)

canvas[..., 0] = (np.angle(f) + np.pi) / (2 * np.pi)
canvas[..., 2] = np.abs(f) / np.max(np.abs(f))
canvas = mpl.colors.hsv_to_rgb(canvas)

ax1.imshow(f.real.T, cmap='seismic', interpolation = 'bicubic', origin = 'lower', extent = xrange + yrange)
ax1.plot([-0.5, 0.5], [0, 0], 'ok')
ax1.set_xlabel("x")
ax1.set_ylabel("y")

ax2.imshow(canvas.transpose(1,0,2), origin = 'lower', extent = xrange + yrange)
ax2.plot([-0.5, 0.5], [0, 0], 'ok')
ax2.set_xlabel("x")
ax2.set_ylabel("y")

plt.tight_layout()
plt.savefig('Interferencia.pdf', dpi = 200, bbox_inches='tight', pad_inches = 0)

In [ ]:
import numpy as np
from scipy.fftpack import fft2, fftshift
import matplotlib.pyplot as plt
import matplotlib as mpl

# Distances are supposed to be in mm (650 nm for red laser)
shape      = (2000, 2000)
sheet_Lx   = 36
sheet_Ly   = 36
radius     = 1
distance   = 5000
wavelength = 6.50e-4
wavenumber = 2 * np.pi / wavelength

x = np.linspace(-sheet_Lx / 2, +sheet_Lx / 2, shape[0] + 1)[:-1, None]
y = np.linspace(-sheet_Ly / 2, +sheet_Ly / 2, shape[1] + 1)[None, :-1]

screen_Lx = 0.5 * distance * wavelength * shape[0] / sheet_Lx
screen_Ly = 0.5 * distance * wavelength * shape[1] / sheet_Ly
extent    = ( - screen_Lx, screen_Lx * (1 - 2/shape[0]) , - screen_Ly, screen_Ly * (1 - 2/shape[1]) )

fig, axes = plt.subplots(1, 2, figsize = (12, 6))
plt.rc('font', size = 15)

cmap      = mpl.colors.LinearSegmentedColormap.from_list("", [(0.0, 'black'), (1.0, 'red')])

for i, separation in enumerate([0, 3]):
    
    # Fresnel diffraction formula
    aperture  = ((x - separation/2)**2 + y**2 < radius**2) | ((x + separation/2)**2 + y**2 < radius**2)
    kernel    = np.exp(1j * wavenumber / (2 * distance) * (x**2 + y**2) )
    amplitude = fftshift(fft2(aperture * kernel))
    power     = np.abs(amplitude)**2
    power    /= power[shape[0]//2, shape[1]//2]
    
    # Plotting
    axes[i].axis('off')
    axes[i].set_xlim(-5,5)
    axes[i].set_ylim(-5,5)
    axes[i].imshow(power.T, cmap = cmap, norm = mpl.colors.PowerNorm(gamma=0.3), interpolation = "bilinear", origin = 'lower', extent = extent)

plt.tight_layout()
plt.savefig('Difraccio.pdf', dpi = 200, bbox_inches='tight', pad_inches = 0)